# Modern LLM Concepts — The Library Version

Four parts, honestly framed: (1) the Bradley-Terry gradient verified numerically; (2) the nucleus-sampling invariant checked; (3) the scale ledger — our miniatures against the real pipeline's numbers; (4) the translation: every from-scratch block mapped to the production stack — shown, not run.

In [1]:
import numpy as np

# (1) Bradley-Terry: dL/dmargin should be −σ(−margin) — check numerically
rs = np.random.default_rng(0)
margin = rs.normal(0, 2, 500)
L = lambda mg: np.log(1 + np.exp(-mg))
analytic = -1 / (1 + np.exp(margin))
eps = 1e-6
numeric = (L(margin + eps) - L(margin - eps)) / (2 * eps)
print(f"Bradley-Terry gradient check: max |analytic − numeric| = {np.abs(analytic - numeric).max():.2e}")
print("dL/dmargin = −σ(−margin): the (guess − truth) of preference — push the gap open in")
print("proportion to how often the judge would currently get the pair wrong.")

Bradley-Terry gradient check: max |analytic − numeric| = 4.73e-10
dL/dmargin = −σ(−margin): the (guess − truth) of preference — push the gap open in
proportion to how often the judge would currently get the pair wrong.


In [2]:
# (2) nucleus (top-p) invariant: the kept set is the SMALLEST prefix reaching mass p
def top_p_mask(pr, p):
    order = np.argsort(pr)[::-1]
    srt = pr[order]
    keep_sorted = np.cumsum(srt) - srt < p
    mask = np.zeros_like(pr); mask[order] = keep_sorted
    return mask.astype(bool)

rs = np.random.default_rng(1)
ok = True
for _ in range(2000):
    z = rs.normal(0, 2, 15)
    pr = np.exp(z - z.max()); pr /= pr.sum()
    mask = top_p_mask(pr, 0.9)
    kept = pr[mask].sum()
    order = np.argsort(pr)[::-1]
    k = mask.sum()
    smaller = pr[order[:k-1]].sum() if k > 1 else 0.0
    ok &= (kept >= 0.9 - 1e-12) and (smaller < 0.9)
print(f"2000 random distributions: kept mass ≥ p AND dropping any one token falls below p -> {ok}")
print("Adaptive truncation: 1 token kept when the model is certain, many when it isn't —")
print("why top-p beats fixed top-k as the production default.")

2000 random distributions: kept mass ≥ p AND dropping any one token falls below p -> True
Adaptive truncation: 1 token kept when the model is certain, many when it isn't —
why top-p beats fixed top-k as the production default.


In [3]:
# (3) the scale ledger
print("OUR MINIATURES vs THE REAL PIPELINE — same recipes, different planets:")
print(f"{'':26s} {'this lesson':>22s} {'production (typ.)':>26s}")
rows = [("pretraining corpus", "8,000 sequences", "trillions of tokens"),
        ("base model", "45,808 params", "10^10 – 10^12 params"),
        ("SFT data", "300 examples", "10^4 – 10^6 dialogues"),
        ("preference pairs", "~4,000 (self-labeled)", "10^5 – 10^6 (human/AI-labeled)"),
        ("reward optimization", "best-of-8", "PPO / DPO / GRPO + KL leash"),
        ("ICL demonstrations", "0–3 shots, T=16", "many-shot, 10^5+ token contexts"),
        ("scaling sweep", "4 sizes, 90x", "decades of compute, 10^9 x")]
for r in rows: print(f"{r[0]:26s} {r[1]:>22s} {r[2]:>26s}")
print()
print("Every dynamic we measured — transfer, forgetting, judge-dependence, Goodhart under")
print("pressure, the ICL ladder, capability cliffs under smooth aggregates — is scale-invariant")
print("in KIND. What scale changes is which rungs and which cliffs are affordable.")

OUR MINIATURES vs THE REAL PIPELINE — same recipes, different planets:
                                      this lesson          production (typ.)
pretraining corpus                8,000 sequences        trillions of tokens
base model                          45,808 params       10^10 – 10^12 params
SFT data                             300 examples      10^4 – 10^6 dialogues
preference pairs            ~4,000 (self-labeled) 10^5 – 10^6 (human/AI-labeled)
reward optimization                     best-of-8 PPO / DPO / GRPO + KL leash
ICL demonstrations                0–3 shots, T=16 many-shot, 10^5+ token contexts
scaling sweep                        4 sizes, 90x decades of compute, 10^9 x

Every dynamic we measured — transfer, forgetting, judge-dependence, Goodhart under
pressure, the ICL ladder, capability cliffs under smooth aggregates — is scale-invariant
in KIND. What scale changes is which rungs and which cliffs are affordable.


### (4) The translation — every block, in the production stack (shown, not run)

```python
# Block 2-4 (pretrain + SFT + mixing) — HuggingFace TRL:
from trl import SFTTrainer                      # next-token loss on curated data = Block 3
trainer = SFTTrainer(model, train_dataset=mixed_instructions)   # 'mixed' = Block 4's replay

# Block 6 (Bradley-Terry) — RewardTrainer consumes (chosen, rejected) pairs:
from trl import RewardTrainer                   # loss: -log σ(r_chosen − r_rejected) = §3.2
rm = RewardTrainer(model=reward_model, train_dataset=preference_pairs)

# Block 7's dynamics, internalized into weights — RLHF with the KL leash:
from trl import PPOTrainer                      # maximize RM score − β·KL(policy ‖ base)
# ...or skip the explicit RM: DPO folds Blocks 6+7 into one loss on the pairs directly:
from trl import DPOTrainer                      # the one-page derivation away from §3.2

# Block 5 (sampling) — every serving stack:
model.generate(prompt, temperature=0.8, top_p=0.9)          # the measured trade, productionized
# best-of-N with a verifier = Block 7's oracle row, at scale (math/code RLVR pipelines)

# Block 8-9 (ICL, scaling) — no code: ICL is prompting; scaling laws are the planning tool
```

Every trainer above minimizes a loss you derived by hand in this course. The rest — data pipelines, distributed compute, safety evaluation — is engineering built on exactly these gradients.

**The course ends here. The library notebooks in all seventeen folders hand you the on-ramp; the from-scratch notebooks are the proof you never have to take any of it on faith.**